# Direct Download - Colab Template Launcher

Minimal Colab runtime bridge notebook for downloading files directly to Google Drive via `aria2` JSON-RPC.

Configure `REPO_OWNER` if using a fork, then run all cells.

In [ ]:
# Cell 1: Mount Google Drive & setup template library
import os
import subprocess
import sys
from google.colab import drive

drive.mount('/content/drive')

REPO_OWNER = 'YOUR_USERNAME'
REPO_URL = f'https://github.com/{REPO_OWNER}/colab-template-library.git'
REPO_DIR = '/content/colab-template-library'

if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=False)
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

sys.path.insert(0, REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'templates', 'direct-download'))

from runtime.colab_bridge import write_runtime_status
from downloader import install_aria2

install_aria2()
write_runtime_status()
print('Runtime bridge ready and Google Drive mounted.')

In [ ]:
# Cell 2: Process pending Direct Download tasks from Google Drive bridge
import json
import time
from runtime.colab_bridge import write_runtime_status
from downloader import download_file, sanitize_destination

TASK_DIR = '/content/drive/MyDrive/.colab-tasks'
os.makedirs(TASK_DIR, exist_ok=True)

print('Listening for pending Direct Download tasks... (Interrupt kernel to stop)')
try:
    while True:
        write_runtime_status(TASK_DIR)
        for tid in sorted(os.listdir(TASK_DIR)):
            status_file = os.path.join(TASK_DIR, tid, 'status.json')
            if not tid.startswith('direct-download_') or not os.path.isfile(status_file):
                continue
            with open(status_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            if data.get('status') == 'pending':
                inputs = data.get('inputs', {})
                print(f'Executing task {tid}...')
                dest = sanitize_destination(inputs.get('destination', '/MyDrive/Downloads'))
                result = download_file(
                    url=inputs['url'],
                    task_id=tid,
                    filename=inputs.get('filename'),
                    destination=dest,
                )
                print(json.dumps(result, indent=2))
        time.sleep(3)
except KeyboardInterrupt:
    print('Bridge loop stopped.')